# PDB Promote (Phase 3)

Promotes staged PDB files from the CTS output prefix into the final
Lakehouse S3 location, archives obsoleted and updated entries, and
trims the transfer manifest for resumability.

## Prerequisites

- Phase 1 (`pdb_manifest.ipynb`) has been run and manifests are available
- Phase 2 (`pdb_rsync_sync` CTS job) has completed and output is in `STAGING_KEY_PREFIX`

## What this notebook does

1. Optionally archives objects for entries in `updated_manifest.txt`
   (these will be overwritten by the promoted files)
2. Archives and deletes objects for entries in `removed_manifest.txt`
   (these are obsolete and should be removed from the live Lakehouse)
3. Promotes every `raw_data/…` file from the staging prefix to the
   final Lakehouse path
4. Trims `transfer_manifest.txt` to remove successfully promoted entries

## Path formats quick reference

| Suffix | Format | Example |
|--------|--------|---------|
| `_URI` | `s3://bucket/key/…` | `s3://cdm-lake/staging/run1/` |
| `_BUCKET` | bucket name only | `cdm-lake` |
| `_KEY_PREFIX` | S3 key prefix (no scheme/bucket) | `tenant-general-warehouse/kbase/datasets/pdb/` |
| `_PATH` | local filesystem path | `output/removed_manifest.txt` |

Staging prefix: `s3://{BUCKET}/{STAGING_KEY_PREFIX}raw_data/<hash>/<pdb_id>/…`
Lakehouse path: `s3://{BUCKET}/{LAKEHOUSE_KEY_PREFIX}raw_data/<hash>/<pdb_id>/…`

In [ ]:
"""Imports."""

from __future__ import annotations

import json
from pathlib import Path

from cdm_data_loaders.pdb.entry import DEFAULT_LAKEHOUSE_KEY_PREFIX
from cdm_data_loaders.pdb.promote import promote_from_s3

In [ ]:
"""Configure parameters."""

# S3 bucket name (no s3:// scheme)
BUCKET = "cdm-lake"

# S3 key prefix where Phase 2 (CTS) wrote its output.
# format: S3 key prefix (no scheme, no bucket), e.g. "staging/pdb-run-2024-01-15/"
STAGING_KEY_PREFIX = "staging/<run-id>/"  # TODO: set for this run

# S3 key prefix for the final Lakehouse location (default is fine for production)
LAKEHOUSE_KEY_PREFIX = DEFAULT_LAKEHOUSE_KEY_PREFIX

# PDB release date tag for archive metadata (YYYY-MM-DD)
PDB_RELEASE = "2024-01-15"  # TODO: set to the Wednesday release date

# Local paths to the manifests produced by Phase 1.
# Download these from S3 before running, or set to None to skip archiving.
REMOVED_MANIFEST_PATH: str | None = "output/removed_manifest.txt"
UPDATED_MANIFEST_PATH: str | None = "output/updated_manifest.txt"

# S3 key of the transfer manifest for trimming after promote.
# Set to None to skip trimming.
MANIFEST_S3_KEY: str | None = f"{STAGING_KEY_PREFIX}transfer_manifest.txt"

# Dry run: log what would be done without making changes
DRY_RUN = True  # Set to False to actually promote

print(f"Bucket: {BUCKET}")
print(f"Staging prefix: {STAGING_KEY_PREFIX}")
print(f"Lakehouse prefix: {LAKEHOUSE_KEY_PREFIX}")
print(f"PDB release: {PDB_RELEASE}")
print(f"Dry run: {DRY_RUN}")

In [ ]:
"""Validate S3 connectivity and staging prefix."""

import boto3
from botocore.exceptions import ClientError, NoCredentialsError

s3 = boto3.client("s3")

try:
    sts = boto3.client("sts")
    identity = sts.get_caller_identity()
    print(f"✓ Credentials valid — account: {identity['Account']}, arn: {identity['Arn']}")
except NoCredentialsError:
    print("✗ No AWS credentials found")
    raise
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidParameterValue":
        print("✓ Credentials present (STS not supported on this endpoint — skipping identity check)")
    else:
        print(f"✗ Credential check failed: {e}")
        raise

try:
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=STAGING_KEY_PREFIX, MaxKeys=5)
    count = resp.get("KeyCount", 0)
    if count > 0:
        print(f"✓ Found {count} object(s) at s3://{BUCKET}/{STAGING_KEY_PREFIX} (+ possibly more)")
        for obj in resp.get("Contents", []):
            print(f"   {obj['Key']}")
    else:
        print(f"✗ No objects found at s3://{BUCKET}/{STAGING_KEY_PREFIX} — check STAGING_KEY_PREFIX")
except ClientError as e:
    print(f"✗ S3 access check failed: {e}")
    raise

In [ ]:
"""Preview manifest file contents before promoting."""

for label, path in [("removed", REMOVED_MANIFEST_PATH), ("updated", UPDATED_MANIFEST_PATH)]:
    if path and Path(path).is_file():
        lines = Path(path).read_text().splitlines()
        print(f"{label}_manifest ({len(lines)} entries): {path}")
        for line in lines[:5]:
            print(f"  {line}")
        if len(lines) > 5:
            print(f"  ... (+{len(lines) - 5} more)")
    elif path:
        print(f"✗ {label}_manifest not found at {path}")
    else:
        print(f"{label}_manifest: not configured (skipping archiving)")

In [ ]:
"""Run promote."""

report = promote_from_s3(
    staging_key_prefix=STAGING_KEY_PREFIX,
    bucket=BUCKET,
    removed_manifest_path=REMOVED_MANIFEST_PATH,
    updated_manifest_path=UPDATED_MANIFEST_PATH,
    pdb_release=PDB_RELEASE,
    manifest_s3_key=MANIFEST_S3_KEY,
    lakehouse_key_prefix=LAKEHOUSE_KEY_PREFIX,
    dry_run=DRY_RUN,
)

print(json.dumps(report, indent=2))

## Post-promote checklist

1. Verify the promote report shows `failed == 0`
2. If `DRY_RUN` was `True`, set it to `False` and re-run the promote cell
3. Spot-check a few promoted entries in the Lakehouse:
   ```
   aws s3 ls s3://cdm-lake/tenant-general-warehouse/kbase/datasets/pdb/raw_data/
   ```
4. Upload the updated `holdings_snapshot.json.gz` to S3 for the next diff